In [1]:
import pandas as pd
import pm4py

print("works")

works


In [2]:
import pm4py

log = pm4py.read_xes("../data/BPI Challenge 2017.xes.gz")

print(log)

/Users/felixhauptmann/ProM-Assignment-Group-C/.venv/lib/python3.13/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

              Action org:resource            concept:name  EventOrigin  \
0            Created       User_1    A_Create Application  Application   
1        statechange       User_1             A_Submitted  Application   
2            Created       User_1          W_Handle leads     Workflow   
3            Deleted       User_1          W_Handle leads     Workflow   
4            Created       User_1  W_Complete application     Workflow   
...              ...          ...                     ...          ...   
1202262      Deleted       User_1     W_Call after offers     Workflow   
1202263      Created       User_1     W_Call after offers     Workflow   
1202264  statechange      User_28             A_Cancelled  Application   
1202265  statechange      User_28             O_Cancelled        Offer   
1202266      Deleted      User_28     W_Call after offers     Workflow   

                       EventID lifecycle:transition  \
0        Application_652823628             complete   
1

In [3]:
df = pm4py.convert_to_dataframe(log)

df.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Polluter Script


### Pattern: Unanchored Events

In [38]:
import numpy as np
import random

In [39]:
OUTPUT_FILE = "BPI_2017_unanchored_events.csv"

In [40]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [41]:
def pollute_mixed_timestamp_formats(df, timestamp_column="time:timestamp", rate = 0.5):
    df = df.copy()

    df[timestamp_column] = df[timestamp_column].astype("object")

    n = int(len(df) * rate)

    indices = np.random.choice(df.index, n, replace=False)

    formats = [
        "%d/%m/%Y %H:%M:%S",   # European format
        "%m/%d/%Y %H:%M:%S",   # US format
        "%Y-%m-%d %H:%M:%S",   # ISO-like format
        "%d-%m-%Y %H:%M:%S",   # ISO-like format german
        "%d.%m.%Y %H:%M:%S",   # German format
        "%d.%m.%Y",   # German format time missing
    ]

    for index in indices:
        timestamp = df.loc[index, timestamp_column]

        if pd.notna(timestamp):
            timestamp = pd.to_datetime(timestamp)
            chosen_formatted = random.choice(formats)
            df.loc[index, timestamp_column] = timestamp.strftime(chosen_formatted)

    return df

In [42]:
def pollute_invalid_timestamps(df, timestamp_column="time:timestamp", rate = 0.5):
    df = df.copy()
    df[timestamp_column] = df[timestamp_column].astype("object")
    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    invalid_values = [
        "2026-14-01 10:00:00",   # 14th month
        "2017-02-30 12:00:00",   # 30th feburary
        "2017-13-01 09:00:00",   # 13th month
        "2017-04-31 15:30:00",   # april with 31 days
        "2017-12-01 25:00:00",   # 25th hour
        "2017-12-01 10:70:00",   # 70 mins
        "01/14/2026 08:00:00",   # not possible with DD/MM/YYYY
        "99/99/9999 99:99:99",
    ]

    df.loc[indices, timestamp_column] = np.random.choice(
        invalid_values,
        size=n
    )

    return df

In [43]:
def pollute_missing_timestamps(df, timestamp_column="time:timestamp", rate = 0.5):
    df = df.copy()
    df[timestamp_column] = df[timestamp_column].astype("object")
    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)
    df.loc[indices, "time:timestamp"] = None
    return df

In [44]:
df_polluted = df.copy()

df_polluted = pollute_mixed_timestamp_formats(df_polluted)
df_polluted = pollute_missing_timestamps(df_polluted)
df_polluted = pollute_invalid_timestamps(df_polluted)

In [45]:
df_polluted.head(100)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,None,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,None,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2017-02-30 12:00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2017-13-01 09:00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Obtained,User_116,W_Validate application,Workflow,Workitem_1121078170,start,None,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,statechange,User_116,A_Validating,Application,ApplState_145257449,complete,13.01.2016,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,statechange,User_116,O_Returned,Offer,OfferState_836197344,complete,None,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_997411923
98,Released,User_116,W_Validate application,Workflow,Workitem_1394551701,suspend,01/13/2016 09:14:02,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
# Check how many different formats the df contains now

s = df_polluted["time:timestamp"].astype(str)

patterns = {
    "ISO format YYYY-MM-DD": r"^\d{4}-\d{2}-\d{2}",
    "European slash DD/MM/YYYY": r"^\d{2}/\d{2}/\d{4}",
    "Dash DD-MM-YYYY or MM-DD-YYYY": r"^\d{2}-\d{2}-\d{4}",
    "US-like MM/DD/YYYY": r"^\d{2}/\d{2}/\d{4}",
    "Invalid / custom strings": r"not_a_timestamp|9999|NaT|None"
}

for name, pattern in patterns.items():
    count = s.str.match(pattern, na=False).sum()
    print(name, count)

ISO format YYYY-MM-DD 626482
European slash DD/MM/YYYY 200090
Dash DD-MM-YYYY or MM-DD-YYYY 24980
US-like MM/DD/YYYY 200090
Invalid / custom strings 0


In [47]:
df_export = df_polluted.copy()

df_export["time:timestamp"] = pd.to_datetime(
    df_export["time:timestamp"],
    errors="coerce",
    utc=True,
)

event_log = pm4py.convert_to_event_log(df_export)
pm4py.write_xes(event_log, "noised.xes")

exporting log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

In [49]:
log_noised = pm4py.read_xes("noised.xes")
df_noised = log_noised
df_noised.head(100)

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_652823628,20000.0
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_652823628,20000.0
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_652823628,20000.0
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_652823628,20000.0
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Existing loan takeover,New credit,Application_652823628,20000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Obtained,User_116,W_Validate application,Workflow,Workitem_1121078170,start,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home improvement,New credit,Application_428409768,15000.0
96,statechange,User_116,A_Validating,Application,ApplState_145257449,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home improvement,New credit,Application_428409768,15000.0
97,statechange,User_116,O_Returned,Offer,OfferState_836197344,complete,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_997411923,Home improvement,New credit,Application_428409768,15000.0
98,Released,User_116,W_Validate application,Workflow,Workitem_1394551701,suspend,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Home improvement,New credit,Application_428409768,15000.0


### Pattern: Polluted Labels

In [18]:
df.head()

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
def pollute_labels(df, rate = 0.5):
    df = df.copy()
    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        activity = df.loc[index, "concept:name"]
        case_id = df.loc[index, "case:concept:name"]
        df.loc[index, "concept:name"] = f"{activity} - Incident No. {case_id}"

    return df


In [20]:
pollute_labels = pollute_labels(df)
pollute_labels.head(100)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted - Incident No. Application_652823628,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads - Incident No. Application_6528...,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads - Incident No. Application_6528...,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Obtained,User_116,W_Validate application - Incident No. Applicat...,Workflow,Workitem_1121078170,start,2016-01-13 08:57:21.342000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,statechange,User_116,A_Validating,Application,ApplState_145257449,complete,2016-01-13 08:57:22.349000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,statechange,User_116,O_Returned,Offer,OfferState_836197344,complete,2016-01-13 09:08:14.217000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Offer_997411923
98,Released,User_116,W_Validate application - Incident No. Applicat...,Workflow,Workitem_1394551701,suspend,2016-01-13 09:14:02.550000+00:00,Home improvement,New credit,Application_428409768,15000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Pattern: Distorted Labels

In [37]:
df["concept:name"].unique()

<StringArray>
[      'A_Create Application',                'A_Submitted',
             'W_Handle leads',     'W_Complete application',
                  'A_Concept',                 'A_Accepted',
             'O_Create Offer',                  'O_Created',
   'O_Sent (mail and online)',        'W_Call after offers',
                 'A_Complete',     'W_Validate application',
               'A_Validating',                 'O_Returned',
    'W_Call incomplete files',               'A_Incomplete',
                 'O_Accepted',                  'A_Pending',
                   'A_Denied',                  'O_Refused',
                'O_Cancelled',                'A_Cancelled',
       'O_Sent (online only)',   'W_Assess potential fraud',
 'W_Personal Loan collection',    'W_Shortened completion ']
Length: 26, dtype: str

In [24]:
df["concept:name"].value_counts()

concept:name
W_Validate application        209496
W_Call after offers           191092
W_Call incomplete files       168529
W_Complete application        148900
W_Handle leads                 47264
O_Create Offer                 42995
O_Created                      42995
O_Sent (mail and online)       39707
A_Validating                   38816
A_Create Application           31509
A_Concept                      31509
A_Accepted                     31509
A_Complete                     31362
O_Returned                     23305
A_Incomplete                   23055
O_Cancelled                    20898
A_Submitted                    20423
O_Accepted                     17228
A_Pending                      17228
A_Cancelled                    10431
O_Refused                       4695
A_Denied                        3753
W_Assess potential fraud        3282
O_Sent (online only)            2026
W_Shortened completion           238
W_Personal Loan collection        22
Name: count, dtype: int64

In [25]:

def distort_activity_label(label):
    replacements = {
        "application": "appl.",
        "Application": "App.",
        "offers": "ofrs.",
        "Offer": "Off.",
        "incomplete": "incompl.",
        "Incomplete": "Incompl.",
        "files": "docs.",
        "leads": "lds.",
        "Handle": "Hndl.",
        "Validate": "Valid.",
        "Validating": "Valid.",
        "Complete": "Comp.",
        "Created": "Crtd.",
        "Create": "Crt.",
        "Accepted": "Acc.",
        "Cancelled": "Canc.",
        "Submitted": "Subm.",
        "Pending": "Pend.",
        "Denied": "Den.",
        "Refused": "Ref.",
        "Returned": "Ret.",
        "potential": "pot.",
        "fraud": "frd.",
        "Shortened": "Short.",
        "completion": "compl.",
        "Personal": "Pers.",
        "Loan": "Ln.",
        "collection": "coll.",
        "online": "onl.",
        "mail": "email"
    }

    distorted = str(label)

    for original, replacement in replacements.items():
        if original in distorted:
            return distorted.replace(original, replacement, 1)

    return distorted + "."

In [36]:
labels = df["concept:name"].dropna().unique()

test_df = pd.DataFrame({
    "original_label": labels,
    "distorted_label": [distort_activity_label(label) for label in labels]
})

test_df

,original_label,distorted_label
0,A_Create Application,A_Create App.
1,A_Submitted,A_Subm.
2,W_Handle leads,W_Handle lds.
3,W_Complete application,W_Complete appl.
4,A_Concept,A_Concept.
5,A_Accepted,A_Acc.
6,O_Create Offer,O_Create Off.
7,O_Created,O_Crtd.
8,O_Sent (mail and online),O_Sent (mail and onl.)
9,W_Call after offers,W_Call after ofrs.


In [28]:
def pollute_distorted_labels(
    df,
    activity_column="concept:name",
    rate=0.05,
    pollution_column="pollution_type"
):
    df = df.copy()

    if pollution_column not in df.columns:
        df[pollution_column] = None

    n = int(len(df) * rate)
    indices = np.random.choice(df.index, n, replace=False)

    for index in indices:
        original_label = df.loc[index, activity_column]

        if pd.notna(original_label):
            distorted_label = distort_activity_label(original_label)

            df.loc[index, activity_column] = distorted_label
            df.loc[index, pollution_column] = "distorted_label"

    return df

In [29]:
df_polluted = pollute_distorted_labels(
    df,
    activity_column="concept:name",
    rate=0.05
)

In [30]:
print("Original unique activities:", df["concept:name"].nunique())
print("Polluted unique activities:", df_polluted["concept:name"].nunique())

df_polluted["pollution_type"].value_counts()

Original unique activities: 26
Polluted unique activities: 52


pollution_type
distorted_label    60113
Name: count, dtype: int64

In [31]:
df_polluted["concept:name"].value_counts()

concept:name
W_Validate application        198888
W_Call after offers           181619
W_Call incomplete files       160037
W_Complete application        141515
W_Handle leads                 44866
O_Created                      40854
O_Create Offer                 40804
O_Sent (mail and online)       37713
A_Validating                   36890
A_Concept                      30018
A_Create Application           29912
A_Accepted                     29897
A_Complete                     29733
O_Returned                     22163
A_Incomplete                   21908
O_Cancelled                    19848
A_Submitted                    19473
O_Accepted                     16409
A_Pending                      16351
W_Validate appl.               10608
A_Cancelled                     9893
W_Call after ofrs.              9473
W_Call incompl. files           8492
W_Complete appl.                7385
O_Refused                       4472
A_Denied                        3571
W_Assess potential fraud 